# Hello World on the portable dev environment

Adapted from the [Lagrange hello-world example](https://gitlab.linksfoundation.com/links-iqm-spark/examples/-/blob/main/01_hello_world.ipynb)
to run against a **portable-deployment** instance (self-signed TLS, dev Keycloak,
mock or real IQM device behind the QC Gateway).


## 1. Install dependencies

No `lagrangeclient` here — the dev Keycloak issues tokens via password grant.


In [ ]:
%pip install "iqm-client[qiskit]==34.0.3" matplotlib requests --quiet


## 2. Configuration

Set the deployment host and fetch the self-signed certificate once.
The test-user password is in `keycloak-deployment/dev-credentials.env` on the host.


In [ ]:
DEV_HOST = "192.0.2.10"  # <-- your deployment host/IP


In [ ]:
# Fetch the self-signed certificate (pick the variant that applies):

# a) notebook running on the deployment host itself:
#!cp ~/workspace/portable-deployment/keycloak-deployment/certs/keycloak.crt ./keycloak.crt

# b) notebook running on your laptop (adjust user/path to the checkout):
!scp ubuntu@{DEV_HOST}:workspace/portable-deployment/keycloak-deployment/certs/keycloak.crt ./keycloak.crt

!ls -l keycloak.crt


In [ ]:
import os, requests, getpass

# Trust the self-signed cert BEFORE anything opens a connection
os.environ["REQUESTS_CA_BUNDLE"] = os.path.abspath("keycloak.crt")

# Get an access token from the dev Keycloak (password grant, dev only)
username = "testuser@example.com"
password = getpass.getpass(f"Password for {username} (see dev-credentials.env): ")
resp = requests.post(
    f"https://{DEV_HOST}:8843/auth/realms/cortex/protocol/openid-connect/token",
    data={"grant_type": "password", "client_id": "test-frontend",
          "username": username, "password": password},
)
resp.raise_for_status()
os.environ["IQM_TOKEN"] = resp.json()["access_token"]
print("token OK — expires in", resp.json()["expires_in"], "s (re-run this cell when it does)")


## 3. Build a circuit


In [ ]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(0, 2)
circuit.measure_all()

print(circuit.draw(output='text'))


## 4. Connect to the gateway

The gateway is API-identical to the machine — unmodified `qiskit-iqm` works.


In [ ]:
from iqm.qiskit_iqm import IQMProvider
from qiskit import transpile
from qiskit.visualization import plot_histogram

iqm_url = f"https://{DEV_HOST}:8000"
provider = IQMProvider(iqm_url, quantum_computer="default")
backend = provider.get_backend()


## 5. Run it

The transpiler maps onto the star topology (two-qubit gates only against the
centre qubit QB3) automatically.


In [ ]:
transpiled_circuit = transpile(circuit, backend=backend)
job = backend.run([transpiled_circuit], shots=1024)
counts = job.result().get_counts()
print(job.job_id())
print(counts)
plot_histogram(counts)


## 6. Where to look next

- **Dashboard**: `https://<DEV_HOST>:8080` — the job appears in the jobs table with
  links to the submitted circuit (object store) and the results viewer; QPU milliseconds are
  billed to the user's default project (`dev-project`).
- To bill a different project, add **after transpilation, before `backend.run()`**:
  `transpiled_circuit.metadata["project"] = "project name"`
  (transpilation deletes metadata).
- The mock device returns placeholder results — perfect correlations are not expected.
